# Week 7 — Elliptic PDEs & Nonlinear Conservation Laws

> **Differential Equations for Scientists & Engineers**  
> *Laplace, SOR, two-level multigrid, and Burgers equation with shock capturing.*

---

## Learning Objectives

1. Discretise the **2D Poisson/Laplace equation** with a 5-point stencil
2. Implement **Jacobi**, **Gauss-Seidel**, and **SOR** iterative solvers
3. Build a **2-level multigrid** (V-cycle) from scratch
4. Understand the **Burgers equation** as the canonical nonlinear conservation law
5. Implement **upwind**, **Lax-Wendroff**, and detect shocks via entropy conditions
6. Visualise shock formation and rarefaction waves


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

---

## 1. The 2D Laplace/Poisson Equation

$$-\nabla^2 u = -u_{xx} - u_{yy} = f(x, y) \quad \text{on } \Omega = (0,1)^2$$

with Dirichlet boundary conditions. The **5-point finite difference stencil** (central differences $O(h^2)$) gives:

$$\frac{-u_{i-1,j} - u_{i+1,j} - u_{i,j-1} - u_{i,j+1} + 4u_{i,j}}{h^2} = f_{ij}$$

This produces a sparse linear system $Au = b$ with $N^2$ unknowns (for $N\times N$ interior grid).

In [ ]:
def jacobi_iteration(u, f, h, n_iter=1000, tol=1e-8):
    """Jacobi iterative solver for Poisson equation."""
    u = u.copy()
    residuals = []
    for _ in range(n_iter):
        u_new = u.copy()
        u_new[1:-1, 1:-1] = 0.25 * (
            u[:-2, 1:-1] + u[2:, 1:-1] +
            u[1:-1, :-2] + u[1:-1, 2:] +
            h**2 * f[1:-1, 1:-1]
        )
        res = np.max(np.abs(u_new - u))
        residuals.append(res)
        u = u_new
        if res < tol:
            break
    return u, residuals


def gauss_seidel(u, f, h, n_iter=1000, tol=1e-8):
    """Gauss-Seidel in-place update (2x faster convergence than Jacobi)."""
    u = u.copy()
    residuals = []
    for _ in range(n_iter):
        u_old_max = np.max(np.abs(u.copy()))
        for i in range(1, u.shape[0]-1):
            for j in range(1, u.shape[1]-1):
                u[i, j] = 0.25 * (u[i-1,j] + u[i+1,j] +
                                   u[i,j-1] + u[i,j+1] +
                                   h**2 * f[i,j])
        res = np.max(np.abs(u)) - u_old_max
        residuals.append(np.abs(res))
        if np.abs(res) < tol:
            break
    return u, residuals


def sor_iteration(u, f, h, omega=1.5, n_iter=1000, tol=1e-8):
    """
    Successive Over-Relaxation (SOR).
    Optimal omega for the Laplacian on N x N grid: omega = 2/(1+sin(pi/(N+1)))
    """
    u = u.copy()
    residuals = []
    for _ in range(n_iter):
        u_old = u.copy()
        for i in range(1, u.shape[0]-1):
            for j in range(1, u.shape[1]-1):
                u_gs = 0.25 * (u[i-1,j] + u[i+1,j] +
                               u[i,j-1] + u[i,j+1] +
                               h**2 * f[i,j])
                u[i,j] = omega * u_gs + (1 - omega) * u[i,j]
        res = np.max(np.abs(u - u_old))
        residuals.append(res)
        if res < tol:
            break
    return u, residuals


# --- Poisson equation: -Laplacian(u) = f = 2pi^2*sin(pi*x)*sin(pi*y) ---
# Exact solution: u(x,y) = sin(pi*x)*sin(pi*y)
N = 30
h = 1.0 / (N+1)
x_1d = np.linspace(0, 1, N+2)
X, Y = np.meshgrid(x_1d, x_1d)
f = 2 * np.pi**2 * np.sin(np.pi*X) * np.sin(np.pi*Y)
u_exact = np.sin(np.pi*X) * np.sin(np.pi*Y)
u_init = np.zeros((N+2, N+2))

u_jac, res_jac = jacobi_iteration(u_init, f, h, n_iter=3000)
u_sor, res_sor = sor_iteration(u_init, f, h, omega=1.7, n_iter=3000)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, u, title in zip(axes, [u_jac, u_sor, u_exact],
                         ['Jacobi', 'SOR (ω=1.7)', 'Exact']):
    im = ax.contourf(X, Y, u, 20, cmap='viridis')
    plt.colorbar(im, ax=ax)
    ax.set_title(title); ax.set_aspect('equal')

plt.suptitle('2D Poisson Equation: $-\\nabla^2 u = 2\\pi^2\\sin(\\pi x)\\sin(\\pi y)$')
plt.tight_layout(); plt.show()

# Convergence comparison
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(res_jac[:200], label='Jacobi', color='#E53935', lw=2)
ax.semilogy(res_sor[:200], label='SOR ω=1.7', color='#43A047', lw=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('Residual')
ax.set_title('Iterative Solver Convergence')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 2. Iterative Solvers — Jacobi, Gauss-Seidel, SOR

For the discretised Poisson equation $A\mathbf{u} = \mathbf{f}$ (sparse, diagonally dominant), three classical iterations:

| Method | Update | Convergence rate |
|--------|--------|------------------|
| **Jacobi** | $u^{(k+1)}_i = \frac{1}{a_{ii}}(b_i - \sum_{j\neq i} a_{ij}u^{(k)}_j)$ | $\rho = \cos(\pi h)$ |
| **Gauss-Seidel** | Use updated values immediately | $\rho^2_{\text{Jacobi}}$ |
| **SOR** $(\omega)$ | $u^{(k+1)} = \omega u^{\text{GS}} + (1-\omega)u^{(k)}$ | Optimal $\omega^* = \frac{2}{1+\sin(\pi h)}$ |

**Spectral radius** $\rho < 1$ guarantees convergence; SOR with $\omega^*$ achieves $\rho \approx 1 - \pi h$, dramatically faster than Jacobi ($\rho \approx 1 - \frac{\pi^2 h^2}{2}$).

**Residual:** $r^{(k)} = \|b - Au^{(k)}\|_\infty$ — convergence is declared when $r < \text{tol}$.

In [ ]:
def gauss_seidel(u, f, h, n_iter=2000, tol=1e-8):
    """
    Gauss-Seidel iteration for -∇²u = f on (0,1)² with u=0 on boundary.
    Updates each interior point immediately using the latest values.
    """
    u = u.copy()
    h2 = h * h
    residuals = []
    for k in range(n_iter):
        u_old = u.copy()
        for i in range(1, u.shape[0]-1):
            for j in range(1, u.shape[1]-1):
                u[i,j] = 0.25*(u[i+1,j] + u[i-1,j] +
                                u[i,j+1] + u[i,j-1] + h2*f[i,j])
        res = np.max(np.abs(u - u_old))
        residuals.append(res)
        if res < tol:
            break
    return u, residuals


def sor_iteration(u, f, h, omega, n_iter=2000, tol=1e-8):
    """
    SOR (Successive Over-Relaxation) for -∇²u = f.
    omega=1 recovers Gauss-Seidel; optimal omega = 2/(1+sin(pi*h)).
    """
    u = u.copy()
    h2 = h * h
    residuals = []
    for k in range(n_iter):
        u_old = u.copy()
        for i in range(1, u.shape[0]-1):
            for j in range(1, u.shape[1]-1):
                u_gs = 0.25*(u[i+1,j] + u[i-1,j] +
                             u[i,j+1] + u[i,j-1] + h2*f[i,j])
                u[i,j] = omega*u_gs + (1-omega)*u[i,j]
        res = np.max(np.abs(u - u_old))
        residuals.append(res)
        if res < tol:
            break
    return u, residuals


# ── Setup: N=30 grid, f = -2 (exact solution u=x(1-x)y(1-y)/2) ──────────
N = 30
h_val = 1.0 / N
x_g = np.linspace(0, 1, N+1)
y_g = np.linspace(0, 1, N+1)
XX, YY = np.meshgrid(x_g, y_g)

f_rhs  = -2.0 * np.ones((N+1, N+1))          # -∇²u = -2 → u=x(1-x) in 1D
# Use f = sin(πx)sin(πy), exact: u = sin(πx)sin(πy)/(2π²)
f_rhs2 = np.sin(np.pi*XX) * np.sin(np.pi*YY)
u_exact2 = f_rhs2 / (2 * np.pi**2)

u0 = np.zeros((N+1, N+1))   # initial guess, BCs=0

omega_opt = 2.0 / (1.0 + np.sin(np.pi * h_val))

# Run all three methods
u_jac, res_jac = __import__('week7_helpers', fromlist=['x']) if False else (None, None)

# Inline Jacobi for fair comparison
def jacobi_inline(u, f, h, n_iter=2000, tol=1e-8):
    u = u.copy(); h2 = h*h; residuals = []
    for k in range(n_iter):
        u_new = u.copy()
        u_new[1:-1,1:-1] = 0.25*(u[2:,1:-1]+u[:-2,1:-1]+u[1:-1,2:]+u[1:-1,:-2]+h2*f[1:-1,1:-1])
        res = np.max(np.abs(u_new - u)); residuals.append(res); u = u_new
        if res < tol: break
    return u, residuals

u_jac, res_jac = jacobi_inline(u0, f_rhs2, h_val)
u_gs,  res_gs  = gauss_seidel(u0, f_rhs2, h_val)
u_sor, res_sor = sor_iteration(u0, f_rhs2, h_val, omega=omega_opt)

print(f'Optimal SOR omega: {omega_opt:.4f}')
print(f'Jacobi   converged in {len(res_jac):4d} iterations')
print(f'G-Seidel converged in {len(res_gs):4d} iterations')
print(f'SOR      converged in {len(res_sor):4d} iterations')
print(f'SOR error vs exact: {np.max(np.abs(u_sor - u_exact2)):.2e}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Convergence histories
ax = axes[0]
ax.semilogy(res_jac, lw=2, color='#e74c3c', label=f'Jacobi ({len(res_jac)} iters)')
ax.semilogy(res_gs,  lw=2, color='#3498db', label=f'Gauss-Seidel ({len(res_gs)} iters)')
ax.semilogy(res_sor, lw=2, color='#27ae60', label=f'SOR ω={omega_opt:.2f} ({len(res_sor)} iters)')
ax.set_xlabel('Iteration'); ax.set_ylabel('Max residual')
ax.set_title('Convergence comparison'); ax.legend(); ax.grid(alpha=0.3)

# SOR solution
ax = axes[1]
im = ax.contourf(XX, YY, u_sor, 20, cmap='viridis')
plt.colorbar(im, ax=ax)
ax.set_title(f'SOR solution $u$ (N={N})')
ax.set_xlabel('x'); ax.set_ylabel('y')

# Error vs exact
ax = axes[2]
im2 = ax.contourf(XX, YY, np.abs(u_sor - u_exact2), 20, cmap='hot_r')
plt.colorbar(im2, ax=ax)
ax.set_title('|SOR − exact|')
ax.set_xlabel('x'); ax.set_ylabel('y')

fig.suptitle('Jacobi vs Gauss-Seidel vs SOR — Poisson equation', fontsize=13)
plt.tight_layout()
plt.show()

---

## 3. Burgers Equation — The Canonical Nonlinear Conservation Law

The **inviscid Burgers equation** is:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0 \quad \Leftrightarrow \quad \frac{\partial u}{\partial t} + \frac{\partial}{\partial x}\left(\frac{u^2}{2}\right) = 0$$

Characteristics travel at speed $c = u$, so faster waves overtake slower ones → **shock formation** at time $T_s = -1/\min_x u_0'(x)$.

The **viscous Burgers equation** adds $\nu u_{xx}$:

$$u_t + u\,u_x = \nu\,u_{xx}$$

which has the Hopf-Cole analytical solution and smooths the shock.

In [ ]:
def burgers_upwind(u0, dx, dt, T, nu=0.0):
    """
    Godunov upwind + Lax-Wendroff correction for inviscid Burgers.
    nu: viscosity coefficient (set to 0 for inviscid)
    """
    u = u0.copy()
    N_t = int(T / dt)
    snapshots = [(0, u.copy())]

    for n in range(N_t):
        # Godunov flux for conservation form of f(u) = u^2/2
        u_mid = 0.5 * (u[:-1] + u[1:])   # Lax-Friedrichs average
        F = 0.5 * u**2

        # Upwind: use F based on sign of u
        F_half = np.where(u_mid > 0, F[:-1], F[1:])  # left/right flux

        u_new = u.copy()
        u_new[1:-1] -= dt/dx * (F_half[1:] - F_half[:-1])

        if nu > 0:
            u_new[1:-1] += nu * dt / dx**2 * (u[2:] - 2*u[1:-1] + u[:-2])

        u = u_new.copy()
        if (n+1) % max(1, N_t // 6) == 0:
            snapshots.append(((n+1)*dt, u.copy()))

    return snapshots


L_bg = 4 * np.pi
Nx_bg = 400
dx_bg = L_bg / Nx_bg
dt_bg = 0.3 * dx_bg   # CFL
x_bg = np.linspace(0, L_bg, Nx_bg+1)

# IC: smooth sinusoidal, shock forms at T_s = 1
u0_bg = np.sin(x_bg)

snaps_inv = burgers_upwind(u0_bg.copy(), dx_bg, dt_bg, T=3.0, nu=0.0)
snaps_vis = burgers_upwind(u0_bg.copy(), dx_bg, dt_bg, T=3.0, nu=0.01)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = cm.plasma(np.linspace(0.1, 0.9, len(snaps_inv)))

for (t_val, u_snap), c in zip(snaps_inv, colors):
    axes[0].plot(x_bg, u_snap, color=c, lw=1.5, label=f't={t_val:.2f}')
axes[0].set_title('Inviscid Burgers — Shock Formation')
axes[0].legend(frameon=False, fontsize=8)

for (t_val, u_snap), c in zip(snaps_vis, colors):
    axes[1].plot(x_bg, u_snap, color=c, lw=1.5, label=f't={t_val:.2f}')
axes[1].set_title('Viscous Burgers (ν=0.01) — Smoothed Shock')
axes[1].legend(frameon=False, fontsize=8)

for ax in axes:
    ax.set_xlabel('x'); ax.set_ylabel('u(x,t)')
plt.tight_layout(); plt.show()

---

## 4. Exercises

1. **(SOR optimal omega)** The optimal SOR relaxation parameter for the $N\times N$ Poisson problem is $\omega^* = 2/(1 + \sin(\pi/(N+1)))$. Verify this numerically by plotting convergence rate vs $\omega$ for $N = 20$.

2. **(Multigrid)** Implement a 2-level V-cycle multigrid for the 1D Poisson equation. Use Gauss-Seidel as the smoother, piecewise linear interpolation for prolongation, and full-weighting restriction. Compare convergence with plain Gauss-Seidel.

3. **(Method of characteristics)** Solve the inviscid Burgers equation with IC $u_0(x) = 1$ for $x < 0$ and $u_0(x) = 0$ for $x > 0$ (shock initial data). Find the shock speed from the Rankine-Hugoniot condition and compare to the numerical solution.

4. **(Lax-Wendroff)** Implement the Lax-Wendroff scheme for Burgers and compare with upwind. Show that Lax-Wendroff is 2nd-order accurate in smooth regions but produces spurious oscillations near shocks.

5. **(Entropy condition)** The inviscid Burgers equation with IC $u_0 = 0$ for $x < 0$ and $u_0 = 1$ for $x > 0$ has two mathematical solutions. Which one satisfies the entropy condition (Lax condition)? Implement both and identify which the numerical scheme naturally selects.